# CURATOR - per-environment cost under vLLM

Settings: **GPU T4 x2**, **Internet On**, the `curator-rl-bundle` dataset attached (new version with
`scripts/kaggle_cost_probe.py`). Expect about 25-40 minutes, most of it the vLLM install.

Why: the pilot timed HF `generate`, which runs each batch until its longest answer ends, so short-answer
environments looked as expensive as long ones. Training generates with vLLM, whose time follows the tokens
actually produced. This measures the real cost spread, with each environment's own token cap (MATH35: 1024).

In [ ]:
# 1) Unpack the bundle

import os, shutil, glob, zipfile
WORK = "/kaggle/working/curator-rl"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
zips = glob.glob("/kaggle/input/**/curator-rl-bundle.zip", recursive=True)
if zips:
    os.makedirs(WORK); zipfile.ZipFile(zips[0]).extractall(WORK)
else:
    roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
    assert roots, "attach the curator-rl-bundle dataset"
    shutil.copytree(roots[0], WORK)
print(open(WORK + "/BUNDLE_INFO.json").read())
assert not os.path.exists(WORK + "/data/test_sealed"), "sealed test set must never be on Kaggle"

In [ ]:
%%bash
# 2) vLLM in its own virtual environment (it pins its own torch; never touch the kernel's)
pip install -q virtualenv
python -m virtualenv -q /kaggle/working/venv_vllm
/kaggle/working/venv_vllm/bin/pip install -q vllm wrapt pydantic pyyaml numpy scipy pandas pyarrow 2>&1 | tail -n 3
/kaggle/working/venv_vllm/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

In [ ]:
%%bash
# 3) The measurement: 0.5B on GPU 0 and 1.5B on GPU 1, in parallel
cd /kaggle/working/curator-rl
mkdir -p /kaggle/working/cost_probe
CUDA_VISIBLE_DEVICES=0 /kaggle/working/venv_vllm/bin/python scripts/kaggle_cost_probe.py \
    --model Qwen/Qwen2.5-0.5B-Instruct --out /kaggle/working/cost_probe > /kaggle/working/cost_probe/log_0p5b.txt 2>&1 &
CUDA_VISIBLE_DEVICES=1 /kaggle/working/venv_vllm/bin/python scripts/kaggle_cost_probe.py \
    --model Qwen/Qwen2.5-1.5B-Instruct --out /kaggle/working/cost_probe > /kaggle/working/cost_probe/log_1p5b.txt 2>&1 &
wait
echo "---- 0.5B ----"; tail -n 14 /kaggle/working/cost_probe/log_0p5b.txt
echo "---- 1.5B ----"; tail -n 14 /kaggle/working/cost_probe/log_1p5b.txt

In [ ]:
# 4) Result tables (copy these back to the chat)
from pathlib import Path
for f in sorted(Path("/kaggle/working/cost_probe").glob("cost_probe_*.md")):
    print(f.read_text())

## 5) GRPO smoke test with the HF backend (kernel Python)
The previous run failed because Kaggle ships `torchao` 0.10, which PEFT 0.20 refuses. Removing it is enough:
nothing here uses torchao.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q trl
%cd /kaggle/working/curator-rl
!CUDA_VISIBLE_DEVICES=0 python scripts/kaggle_trl_probe.py --out /kaggle/working/trl_probe

In [ ]:
# 6) HF-backend GRPO step time (the number still missing from the step probe)
!CUDA_VISIBLE_DEVICES=0 python scripts/kaggle_step_probe.py --out /kaggle/working/step_probe --prompts-per-step 16 --steps 3 2>&1 | tail -n 15